# LIGTAS-pH — Calibration & Verification Notebook

Companion to `generate_dataset.py`. Use this to fill in the parameter
table, check the physics, and produce your Chapter 4 figures.

**Run order:** top to bottom. Re-run section 3 after every parameter change.

| Section | What it gives you |
|---|---|
| 1 | Parameter citation status — your week 1 progress bar |
| 2 | Look at one generated sample |
| 3 | Physics checks (sign + non-triviality) — run after every change |
| 4 | Compare against the real NHSI pork cube |
| 5 | Sensitivity sweep — a Chapter 4 result |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import generate_dataset as G   # single source of truth for the physics

plt.rcParams['figure.dpi'] = 110
print('wavelengths:', G.WAVELENGTHS)

---
## 1. Parameter status

Every `TODO` below is a number you still need to source. Open
`generate_dataset.py`, edit the `PARAMS` dict, save, restart the kernel,
re-run.

Sources: **Piao et al. (2025)** for myoglobin coefficients (open access,
your ref [8]); **omlc.org** for the water absorption table.

In [ ]:
n_todo = G.check_params()
print(f'\n{n_todo} parameters still need a citation.')

---
## 2. Inspect one sample

Sanity check by eye. The pH map should look like a smooth gradient, not
noise. The six bands should differ from each other — if they look
identical, the myoglobin coefficients aren't doing anything.

In [ ]:
import os
os.makedirs('_scratch', exist_ok=True)
rng = np.random.default_rng(7)
cube, ph, mask = G.generate_sample('demo', '_scratch', rng)

fig, ax = plt.subplots(2, 4, figsize=(15, 7))
for i, w in enumerate(G.WAVELENGTHS):
    a = ax.flat[i]
    im = a.imshow(np.where(mask, cube[:, :, i], np.nan), cmap='gray')
    a.set_title(f'{int(w)} nm'); a.axis('off')
    plt.colorbar(im, ax=a, fraction=0.046)

a = ax.flat[6]
im = a.imshow(np.where(mask, ph, np.nan), cmap='turbo')
a.set_title('true pH (hidden label)'); a.axis('off')
plt.colorbar(im, ax=a, fraction=0.046)

rgb = np.dstack([cube[:,:,3], cube[:,:,1], cube[:,:,0]])
ax.flat[7].imshow(np.clip(rgb/rgb.max(), 0, 1))
ax.flat[7].set_title('pseudo-RGB (600/525/481)'); ax.flat[7].axis('off')
plt.tight_layout(); plt.show()

print(f'pH range in this sample: {np.nanmin(ph):.2f} to {np.nanmax(ph):.2f}')

---
## 3. Physics checks

**Run this after every parameter edit.** Two things must hold.

### 3a. Sign — reflectance must FALL as pH rises

Low pH → protein denaturation → more scattering → **pale** meat (PSE) →
high reflectance. High pH → **dark** meat (DFD) → low reflectance.

If the curves slope upward, the sign is inverted.

In [ ]:
absorb = G.mu_a(0.25, 0.60, 0.15, G.P('c_Mb_mean'))
ph_axis = np.linspace(5.3, 6.6, 40)
R = np.array([G.kubelka_munk(absorb, G.mu_s_prime(np.array(p))) for p in ph_axis])

plt.figure(figsize=(7, 4.5))
for i, w in enumerate(G.WAVELENGTHS):
    plt.plot(ph_axis, R[:, i], label=f'{int(w)} nm')
plt.xlabel('pH'); plt.ylabel('diffuse reflectance')
plt.title('Reflectance vs pH — all curves must slope DOWN')
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()

ok = np.all(R[-1] < R[0])
print('PASS — reflectance falls with pH' if ok else 'FAIL — sign is inverted')

### 3b. Non-triviality — a linear model must do POORLY

This is the number that protects you at the defense. If plain linear
regression on raw pixels can solve the task, your CNN proves nothing.

**Target: R² well below 0.9.** Report this in Chapter 4 as the baseline
your model has to beat.

In [ ]:
rng = np.random.default_rng(0)
X, y = [], []
for i in range(40):
    c, p, m = G.generate_sample(f'lin_{i}', '_scratch', rng)
    idx = rng.choice(np.flatnonzero(m), 400, replace=False)
    X.append(c.reshape(-1, 6)[idx]); y.append(p.reshape(-1)[idx])
X = np.vstack(X); y = np.concatenate(y)

Xa = np.c_[X, np.ones(len(X))]
coef, *_ = np.linalg.lstsq(Xa, y, rcond=None)
pred = Xa @ coef
r2 = 1 - ((y-pred)**2).sum() / ((y-y.mean())**2).sum()

print(f'Linear baseline R2  = {r2:.4f}')
print(f'Linear baseline MAE = {np.abs(y-pred).mean():.4f} pH units')
print()
print('GOOD — a CNN has real work to do' if r2 < 0.9
      else 'WARNING — too easy. Widen the myoglobin nuisance ranges.')

plt.figure(figsize=(4.5, 4.5))
plt.scatter(y, pred, s=1, alpha=.15)
lim = [y.min(), y.max()]
plt.plot(lim, lim, 'r--', lw=1)
plt.xlabel('true pH'); plt.ylabel('linear prediction')
plt.title(f'Linear baseline (R2={r2:.3f})'); plt.grid(alpha=.3); plt.show()

---
## 4. Compare against the real NHSI cube

The one external reality check you have. Their camera covers 900–1700 nm,
so **970 nm is the only band that overlaps yours**.

Set the path below to a downloaded pork cube.

In [ ]:
NHSI_PATH = None   # e.g. 'data/nhsi/pork_t01.mat'

if NHSI_PATH is None:
    print('Set NHSI_PATH to a downloaded cube, then re-run this cell.')
else:
    from extract_sensor_params import load_cube, orient, find_meat
    real = orient(load_cube(NHSI_PATH)).astype(float)
    if real.max() > 1.5:
        print('WARNING: values exceed 1.0 — likely raw counts, not')
        print('reflectance. The 970 nm comparison is not valid until')
        print('you apply their white/dark references.')
    rmask = find_meat(real)

    wl = np.linspace(900, 1700, real.shape[2])
    i970 = int(np.argmin(np.abs(wl - 970)))
    real_970 = real[:, :, i970][rmask]

    sim_970 = cube[:, :, 5][mask]

    print(f'real  970nm: mean {real_970.mean():.4f}  sd {real_970.std():.4f}')
    print(f'sim   970nm: mean {sim_970.mean():.4f}  sd {sim_970.std():.4f}')

    plt.figure(figsize=(7, 4))
    plt.hist(real_970, bins=60, alpha=.6, density=True, label='real (NHSI)')
    plt.hist(sim_970, bins=60, alpha=.6, density=True, label='simulated')
    plt.xlabel('reflectance at ~970 nm'); plt.ylabel('density')
    plt.title('970 nm: simulated vs real pork  [CHAPTER 4 FIGURE]')
    plt.legend(); plt.grid(alpha=.3); plt.show()

---
## 5. Sensitivity sweep — a Chapter 4 result

`denat_amplitude` controls how strongly pH affects scattering. It is your
**weakest assumption** — the direction is well established, the magnitude
is not.

Do not pin one value. Sweep it and report how results change. That turns
your softest point into a reported finding.

In [ ]:
original = G.PARAMS['denat_amplitude']['value']
sweep = [0.2, 0.35, 0.5, 0.65, 0.8]
results = []

for amp in sweep:
    G.PARAMS['denat_amplitude']['value'] = amp
    rng = np.random.default_rng(3)
    Xs, ys = [], []
    for i in range(12):
        c, p, m = G.generate_sample(f'sw_{i}', '_scratch', rng)
        idx = rng.choice(np.flatnonzero(m), 400, replace=False)
        Xs.append(c.reshape(-1,6)[idx]); ys.append(p.reshape(-1)[idx])
    Xs = np.vstack(Xs); ys = np.concatenate(ys)
    Xa = np.c_[Xs, np.ones(len(Xs))]
    cf, *_ = np.linalg.lstsq(Xa, ys, rcond=None)
    pr = Xa @ cf
    results.append(1 - ((ys-pr)**2).sum()/((ys-ys.mean())**2).sum())
    print(f'  denat_amplitude={amp:.2f}  linear R2={results[-1]:.4f}')

G.PARAMS['denat_amplitude']['value'] = original

plt.figure(figsize=(6, 4))
plt.plot(sweep, results, 'o-')
plt.xlabel('denaturation amplitude (assumed)')
plt.ylabel('linear baseline R2')
plt.title('Sensitivity to the pH-scattering assumption  [CHAPTER 4 FIGURE]')
plt.grid(alpha=.3); plt.show()

print('\nInterpretation: a stronger assumed pH-scattering coupling makes')
print('the task easier. Report this range rather than one number, and say')
print('plainly that the true coupling strength is not established.')

---
## 6. Generate the full dataset

Once every parameter is cited and both checks in section 3 pass, run from
a terminal (not here — it writes 400 samples):

```bash
python generate_dataset.py --n 400 --out ligtas_synthetic_dataset
```

Then train on the **4 sparse points only**, keeping `phtrue.npy` hidden
until evaluation. That separation is the core experiment.

In [ ]:
import shutil
shutil.rmtree('_scratch', ignore_errors=True)
print('scratch files cleared')